# Experiment E03 — Dictionary Efficiency (η), PreT and SigT

> **Experiment Specification / Experiment Contract**

## Experiment Identity

- **Experiment ID:** `E03`
- **Experiment title:** Dictionary efficiency (η), PreT and SigT
- **Scientific purpose:** Analyse the main-sweep results to establish the
  *dictionary-efficiency* and *statistical-dominance* arguments for the
  shared federated dictionary. Backfills the dictionary efficiency
  `η = (1 − NMSE) / NNZC · 100` into `RESULTS`, computes the SIBS-paper
  Preference Test (PreT) over `{SSIM, PSNR, η}`, and the explicit pairwise
  significance matrix (SigT) on SSIM.
- **Notebook source:** `E03_dictionary_efficiency_preT_sigT.ipynb`.
- **Run folder:** `F_SIBS_r_<date>_<n>/` (shared with E00/E01/E02).
- **Outputs (this enhanced notebook):**
  - `RUN_DIR/figures/exp03_pret_heatmap.png`
  - `RUN_DIR/figures/exp03_efficiency_vs_quality.png`
  - `RUN_DIR/figures/exp03_efficiency_bar_chart.png`
  - `RUN_DIR/figures/exp03_sigt_matrix_ssim.png`
  - `RUN_DIR/Exp03_Results.tex` (consolidated LaTeX results file)

## Scientific Objective

- **Objective:**
  1. Backfill the dictionary efficiency `η` into `RESULTS`.
  2. Compute PreT over `{SSIM, PSNR, η}` for every method in `RESULTS`.
  3. Compute the explicit pairwise binary significance matrix (SigT) on SSIM.
  4. Visualise the η–SSIM Pareto trade-off, the mean-η bar chart, the
     PreT heat-map, and the SigT clustered heat-map.
- **Research question(s):**
  1. Does the federated F-SIBS dictionary achieve higher η than the local
     SIBS dictionaries at equal quality?
  2. Does any single method dominate all others on every metric (PreT = 1.0)?
  3. What is the pairwise dominance structure on SSIM?
- **Hypothesis:** The federated F-SIBS variants achieve higher η than the
  local SIBS variants at equal SSIM, and dominate (or are competitive
  with) the codec baselines on the η–SSIM Pareto front.

## Data

- **Dataset / data source:** Same as E01. E03 re-uses E01's main-sweep cache
  when the sweep settings match, and otherwise computes the sweep itself.
- **Relevant assumptions:** `RESULTS` carries `dataset, method, config,
  node, k, BPP, SSIM, PSNR, NMSE`. The η-backfill cell recomputes
  `NNZC = k · w` and `η = (1 − NMSE) / NNZC · 100` directly if missing.

## Methods

- **Proposed methods:** `F-SIBS_1..4`.
- **Baselines:** `OMP`, `SIBS1..4`.
- **Reference methods:** same as E01.
- **Method configurations:** Identical to E01. E03 does not change the
  sweep configuration; it only analyses `RESULTS`.

## Experimental Design

- **Dependent variables (this experiment):**
  - `η` (dictionary efficiency, %), where `η = (1 − NMSE) / NNZC · 100`,
    `NNZC = k · w` for column-wise sparse coding.
  - `PreT(method, metric)` in [0, 1].
  - `SigT(A_j, A_l)` in {0, 1}.
- **Controlled variables:** α = 0.05 (Welch t-test threshold).
- **Statistical analysis:** Welch independent-samples t-test (unequal
  variance, unequal N). Distinct from E01's paired t-test because RESULTS
  rows for different methods are not generally paired 1:1 across families.

## Evaluation

- **Metrics:** η (dictionary efficiency, %), PreT, SigT.
- **Mathematical definitions:**
  - `η = (1 − NMSE) / NNZC · 100`, `NNZC = k · w`.
  - `PreT(A_j, metric) = mean_{l ≠ j} SigT(A_j, A_l, metric)`.
  - `SigT(A_j, A_l, metric) = 1` iff `Welch_ttest.p ≤ α` and `mean(A_j) > mean(A_l)`.
- **Aggregation:** η per row of RESULTS; PreT and SigT across all rows for
  the given metric.

## Outputs

- **Raw experimental results:** `RESULTS` (with backfilled `NNZC`, `η`).
- **Processed results:** `EFFICIENCY_SUMMARY_TABLE`, `_pret_wide`, `SIGT_SSIM`.
- **Figures (PNG @ 300 DPI, shared `figures/` folder):**
  - `exp03_pret_heatmap.png`, `exp03_efficiency_vs_quality.png`,
    `exp03_efficiency_bar_chart.png`, `exp03_sigt_matrix_ssim.png`
- **Tables:** `EFFICIENCY_SUMMARY_TABLE`, `_pret_wide`, `SIGT_SSIM`.
- **Consolidated LaTeX:** `RUN_DIR/Exp03_Results.tex`.

## Reproducibility

- **Software environment:** Python 3.12.12. Captured at run time.
- **Hardware:** Linux x86_64, 96 CPU cores.
- **Random seeds:** `RNG_SEED` (global).
- **Execution configuration:** `RUN_DIR_POLICY = "latest"`; same selection as E01.
- **`[REQUIRES CONFIRMATION]` items:**
  - `fsibs.protocol.RNG_SEED` (exact integer value).
  - Whether `RESULTS` is loaded from E01's cache or recomputed by E03.


# E03 -- Dictionary efficiency (eta), PreT and SigT

Analyses the main sweep: backfills NNZC / eta, computes the preference test (PreT) with bootstrap CIs, the eta-SSIM Pareto scatter and the explicit pairwise significance matrix (SigT).

* **Independent:** run this notebook top-to-bottom in a fresh kernel; it needs no other notebook's in-memory state.
* **Needs:** the main sweep table. It is obtained with `fsibs.main_sweep.load_or_run_main_sweep`, which re-uses E01's cached CSV when the datasets / algorithms / protocol values below match E01's, and otherwise computes the sweep itself -- so this notebook also runs on its own.
* **Writes:** `sibs_exp3_*` tables and figures, `results/experiment_3/`.
* **Shared code:** imported from the `fsibs` package (see `README.md`); everything experiment-specific is in this notebook.

> **Keep the sweep settings identical to E01** (datasets, algorithms, `K_SWEEP`, `K_SUMMARY`, `DELTA`, `K_GLOBAL`, `N_ROUNDS_SWEEP`) to re-use its cache instead of recomputing.

In [5]:
# ============================================================
# F-SIBS MODULAR - PROJECT LOCATION + IMPORTS
# ============================================================

import sys,time, os
from pathlib import Path

# ------------------------------------------------------------
# Project location
# ------------------------------------------------------------

PROJECT_ROOT = Path("/home/ahmed_omara/F_SIBS_Modular").resolve()
PACKAGE_ROOT = PROJECT_ROOT / "fsibs"

# Validate project root
if not PROJECT_ROOT.exists():
    raise FileNotFoundError(
        f"\nF-SIBS project directory was not found:\n"
        f"  {PROJECT_ROOT}"
    )

# Validate fsibs package
if not PACKAGE_ROOT.exists():
    raise FileNotFoundError(
        f"\nF-SIBS package directory was not found:\n"
        f"  {PACKAGE_ROOT}\n\n"
        f"Project contents:\n"
        + "\n".join(f"  {p.name}" for p in PROJECT_ROOT.iterdir())
    )

if not (PACKAGE_ROOT / "__init__.py").exists():
    raise FileNotFoundError(
        f"\nF-SIBS package exists, but __init__.py was not found:\n"
        f"  {PACKAGE_ROOT / '__init__.py'}"
    )

# Add project root to Python import path
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Use F-SIBS project as working directory
import os
os.chdir(PROJECT_ROOT)

# ------------------------------------------------------------
# Standard imports
# ------------------------------------------------------------
from fsibs.datasets import build_data_bundle
from fsibs.env import setup_environment
from fsibs.io_utils import save_figure, save_table
from fsibs.main_sweep import load_or_run_main_sweep
from fsibs.parallel import describe
from fsibs.runtime import init_run, new_manifest, save_manifest
from fsibs.selection import prompt_selection, resolve_selection
from fsibs.stats import HAS_SCIPY, stats

ModuleNotFoundError: No module named 'fsibs'

## Configuration
All settings of this experiment.

In [1]:
# ------------------------------------------------------------------------------------
# EXPERIMENT CONFIGURATION -- everything that can differ between experiments lives HERE,
# in this notebook.  Edit freely; no other notebook or module is affected.
# ------------------------------------------------------------------------------------
RUN_DIR_POLICY = "latest"      # "latest": re-use the newest F_SIBS_r_<date>_<n> folder (shared with the other
                               #   notebooks) | "new": start a fresh run folder | or an explicit folder path
INTERACTIVE_SELECTION = False  # True: ask for datasets / algorithms with the original input() menus

# Datasets: any of "coil20", "coil100", "eth80", "epfl", "dino", "temple"
SELECTED_DATASETS = ["coil20", "dino", "temple"]
# Algorithms: SIBSk and F-SIBS_k must be selected together (pairing rule); all other baselines are free
SELECTED_ALGORITHMS = [
    "OMP",
    "SIBS1",
    "F-SIBS_1",
    "SIBS2",
    "F-SIBS_2",
    "SIBS3",
    "F-SIBS_3",
    "SIBS4",
    "F-SIBS_4",
    "NDIC-equiv s=4",
    "NDIC-equiv s=8",
    "NDIC-real",
    "LDMIC-equiv",
    "BPG-equiv (JPEG2000)",
    "JPEG",
    "JPEG-LS",
    "SPIHT",
    "CALIC",
]

# ---- shared evaluation-protocol values (defaults from fsibs.protocol; override here for THIS experiment only) ----
from fsibs import protocol as P
RNG_SEED = P.RNG_SEED
COIL_VIEWS = P.COIL_VIEWS
COIL100_VIEWS = P.COIL100_VIEWS
ETH80_VIEWS = P.ETH80_VIEWS
COIL100_MAX_OBJECTS = P.COIL100_MAX_OBJECTS
ETH80_MAX_OBJECTS_PER_CAT = P.ETH80_MAX_OBJECTS_PER_CAT
EPFL_MAX_FRAMES = P.EPFL_MAX_FRAMES
EPFL_TARGET_HW = P.EPFL_TARGET_HW
COIL_N_CROSS_GROUPS = P.COIL_N_CROSS_GROUPS
K_SUMMARY = P.K_SUMMARY
K_GLOBAL = P.K_GLOBAL
N_ROUNDS_SWEEP = P.N_ROUNDS_SWEEP
DELTA = P.DELTA
K_SWEEP = P.K_SWEEP
GEOMETRY = P.GEOMETRY

# ---- settings specific to this experiment ----
FORCE_RECOMPUTE = False                              # E1 main sweep cache
USE_PARALLEL_SWEEP = True                            # E1: flip to False to force serial fallback

ModuleNotFoundError: No module named 'fsibs'

## Setup
Run folder, selection and (when needed) datasets.

In [ ]:
setup_environment(RNG_SEED)                       # warnings, global seed, Matplotlib defaults
describe()                                       # CPU / platform banner
run = init_run(RUN_DIR_POLICY)                    # resolves F_SIBS_r_<date>_<n> and creates its folders
RUN_DIR, FIG_DIR, RESULTS_DIR, DATA_DIR = run.RUN_DIR, run.FIG_DIR, run.RESULTS_DIR, run.DATA_DIR
EXPERIMENT_MANIFEST = new_manifest()              # files this notebook registers as outputs
print("RUN_DIR     :", RUN_DIR)
print("DATA_DIR    :", DATA_DIR)

# ---- algorithm / dataset selection (enforces the SIBSk <-> F-SIBS_k pairing rule) ----
if INTERACTIVE_SELECTION:
    SELECTED_DATASETS, SELECTED_ALGORITHMS = prompt_selection()
sel = resolve_selection(SELECTED_DATASETS, SELECTED_ALGORITHMS)
SELECTED_DATASETS, SELECTED_ALGORITHMS = sel.SELECTED_DATASETS, sel.SELECTED_ALGORITHMS
SELECTED_METHODS = sel.SELECTED_METHODS
SELECTED_DATASETS_SET = sel.SELECTED_DATASETS_SET
ACTIVE_F_SIBS_VARIANTS = sel.ACTIVE_F_SIBS_VARIANTS

# ---- datasets: acquisition (cached in DATA_DIR), node pools, federated groups ----
data = build_data_bundle(
    SELECTED_DATASETS, DATA_DIR,
    coil_views=COIL_VIEWS, coil100_views=COIL100_VIEWS, eth80_views=ETH80_VIEWS,
    coil100_max_objects=COIL100_MAX_OBJECTS, eth80_max_objects_per_cat=ETH80_MAX_OBJECTS_PER_CAT,
    epfl_max_frames=EPFL_MAX_FRAMES, epfl_target_hw=EPFL_TARGET_HW,
    coil_n_cross_groups=COIL_N_CROSS_GROUPS)
POOLS, PHI, GROUPS, CONV_SPECS = data.POOLS, data.PHI, data.GROUPS, data.CONV_SPECS

## Experiment-specific helpers
Defined here (not in `fsibs`) because no other experiment uses them.

In [ ]:
def compute_preT(results_df, metric="SSIM", alpha=0.05, methods=None):
    """SIBS-paper Preference Test (PreT) for `metric` (e.g. 'SSIM', 'PSNR', 'eta').

    Parameters
    ----------
    results_df : DataFrame with at least ["method", metric] columns
                 (typically RESULTS, or a filtered subset of it).
    metric     : column name to compare across methods.
    alpha      : significance level for the pairwise Welch t-test.
    methods    : optional explicit method list/order; default = every
                 distinct method present in results_df with the metric.

    Returns
    -------
    DataFrame with columns [method, metric, PreT, n_comparisons],
    sorted by PreT descending (paper's ranking order).
    """
    sub = results_df[["method", metric]].dropna()
    if methods is None:
        methods = sorted(sub["method"].unique().tolist())
    method_vals = {m: sub.loc[sub["method"] == m, metric].to_numpy(dtype=float)
                  for m in methods}

    sig = {m: {} for m in methods}
    for mj in methods:
        x = method_vals[mj]
        for ml in methods:
            if mj == ml:
                continue
            y = method_vals[ml]
            if len(x) < 2 or len(y) < 2 or not HAS_SCIPY:
                sig[mj][ml] = 0.0
                continue
            if np.std(x) == 0 and np.std(y) == 0:
                # degenerate: identical constants everywhere -> never
                # "significantly higher"; a genuine constant gap is
                # ill-defined for a t-test, so treat conservatively as 0
                sig[mj][ml] = 1.0 if (np.mean(x) > np.mean(y)) else 0.0
                continue
            _, p_val = stats.ttest_ind(x, y, equal_var=False)
            sig[mj][ml] = 1.0 if (p_val <= alpha and np.mean(x) > np.mean(y)) else 0.0

    rows = []
    for mj in methods:
        others = [ml for ml in methods if ml != mj]
        scores = [sig[mj][ml] for ml in others]
        rows.append({"method": mj, "metric": metric,
                     "PreT": float(np.mean(scores)) if scores else float("nan"),
                     "n_comparisons": len(others)})
    summary = (pd.DataFrame(rows)
              .sort_values("PreT", ascending=False)
              .reset_index(drop=True))
    return summary

def compute_sigT_matrix(results_df, metric="SSIM", alpha=0.05, methods=None):
    """Pairwise binary significance matrix SigT(A_j, A_l) in {0, 1}:
    1 iff a Welch t-test on `metric` between A_j and A_l is significant
    at `alpha` AND mean(A_j) > mean(A_l). Diagonal is left as NaN
    (undefined: a method is not compared against itself)."""
    sub = results_df[["method", metric]].dropna()
    if methods is None:
        methods = sorted(sub["method"].unique().tolist())
    vals = {m: sub.loc[sub["method"] == m, metric].to_numpy(dtype=float) for m in methods}
    mat = pd.DataFrame(np.nan, index=methods, columns=methods)
    for mj in methods:
        x = vals[mj]
        for ml in methods:
            if mj == ml:
                continue
            y = vals[ml]
            if len(x) < 2 or len(y) < 2 or not HAS_SCIPY:
                mat.loc[mj, ml] = 0.0
                continue
            if np.std(x) == 0 and np.std(y) == 0:
                mat.loc[mj, ml] = 1.0 if np.mean(x) > np.mean(y) else 0.0
                continue
            _, p_val = stats.ttest_ind(x, y, equal_var=False)
            mat.loc[mj, ml] = 1.0 if (p_val <= alpha and np.mean(x) > np.mean(y)) else 0.0
    return mat

## Experiment

In [ ]:
RESULTS = load_or_run_main_sweep(
    POOLS, PHI, GROUPS,
    selected_datasets=SELECTED_DATASETS, selected_methods=SELECTED_METHODS,
    active_f_sibs_variants=ACTIVE_F_SIBS_VARIANTS,
    k_sweep=K_SWEEP, k_summary=K_SUMMARY, delta=DELTA, k_global=K_GLOBAL,
    n_rounds_sweep=N_ROUNDS_SWEEP, out_dir=FIG_DIR,
    force_recompute=FORCE_RECOMPUTE, use_parallel=USE_PARALLEL_SWEEP)

# ============================================================
# EXPERIMENT 3 - Dictionary efficiency, PreT & SigT (original Cells
# 42, 43, 45, 46, 48 + backfill). Backfills NNZC / eta (dictionary
# efficiency) into RESULTS; runs compute_preT (Cell 2) over
# {SSIM, PSNR, eta}; draws the eta-SSIM Pareto scatter with the
# efficiency bar chart, the PreT bar chart with bootstrap CIs; and the
# explicit SigT pairwise significance matrix (compute_sigT_matrix,
# Cell 2) as clustered heat-map. Tables: PreT +/- CI per metric, full
# SigT with raw p. compute_preT/compute_sigT_matrix live in Cell 2.
# ============================================================

# ============================================================
# Cell 42 (NEW) - Dictionary Efficiency (eta): RESULTS backfill
#   ADDITIVE ONLY. node_metrics (Cell 3) now returns "NNZC"/"eta" for
#   every future sweep run. This cell guarantees the columns are present
#   in RESULTS *right now* too, even if RESULTS was loaded from a cache
#   file written before that update -- it recomputes eta directly from
#   columns the cache already has (NMSE, k, and each row's image width),
#   so no re-run of the (expensive) main sweep is required.
#   Formula (paper): eta = (1 - NMSE) / NNZC * 100, NNZC = k * w for this
#   notebook's column-wise sparse coding (Cell 3 dict_efficiency()).
# ============================================================
_needs_backfill = ("eta" not in RESULTS.columns) or ("NNZC" not in RESULTS.columns) \
    or RESULTS["eta"].isna().all()

if _needs_backfill:
    _w_per_row = RESULTS["dataset"].map(lambda d: GEOMETRY.get(d, (np.nan, np.nan))[1])
    _nnzc = RESULTS["k"].astype(float) * _w_per_row.astype(float)
    RESULTS["NNZC"] = _nnzc
    with np.errstate(invalid="ignore", divide="ignore"):
        _eta = (1.0 - RESULTS["NMSE"].astype(float)) / _nnzc * 100.0
    _eta[~np.isfinite(_nnzc) | (_nnzc <= 0)] = np.nan
    RESULTS["eta"] = _eta
    print("Backfilled NNZC/eta into RESULTS (%d rows; %d have a finite eta)."
          % (len(RESULTS), int(RESULTS["eta"].notna().sum())))
else:
    print("RESULTS already has NNZC/eta (from a sweep run after the node_metrics update); "
          "%d/%d rows have a finite eta." % (int(RESULTS["eta"].notna().sum()), len(RESULTS)))

# ---- quick sanity summary: mean eta per method (single-image methods, k-averaged) ----
_eta_summary = (RESULTS[RESULTS["family"].isin(["single", "federated"]) & RESULTS["eta"].notna()]
                .groupby("method")["eta"].agg(["mean", "std", "count"])
                .sort_values("mean", ascending=False))
print("\nMean Dictionary Efficiency (eta, %) by method (single-image + federated families):")
print(_eta_summary.to_string(float_format=lambda v: "%.4f" % v))

# ============================================================
# Cell 43 (NEW) - SIBS-paper Preference Test (PreT)
#   ADDITIVE ONLY: reuses HAS_SCIPY/stats from Cell 1b but does not touch
#   paired_ttest / onesample_ttest / holm_bonferroni / hierarchical_ttest.
#
#   Definition (paper): for every ORDERED pair of methods (A_j, A_l),
#   A_j != A_l, run a significance test between their per-row metric
#   values. SigT(A_j, A_l) = 1 iff the test is significant at `alpha`
#   AND mean(A_j) > mean(A_l); else 0. PreT(A_j) is the mean of A_j's
#   SigT scores against every other method -- i.e. the fraction of
#   rival methods A_j is *significantly better than*, ranging 0..1.
#
#   Test choice: RESULTS rows for different methods are not generally
#   paired 1:1 (different families have different row counts -- e.g. a
#   codec's rate sweep vs. a fixed-k method), so this uses an
#   independent-samples Welch t-test (unequal variance, unequal N) rather
#   than the notebook's existing per-node *paired* test. This is a
#   distinct (unpaired, many-methods) comparison from the paired/
#   hierarchical tests in Cell 1b/8b and does not replace them.
# ============================================================

# ---- sanity run: PreT on SSIM for whatever is currently in RESULTS --------
_preT_demo_methods = [m for m in RESULTS["method"].unique()
                      if RESULTS.loc[RESULTS["method"] == m, "SSIM"].notna().any()]
if _preT_demo_methods:
    print("PreT(SSIM) ranking (all methods currently in RESULTS):")
    print(compute_preT(RESULTS, metric="SSIM", alpha=globals().get("ALPHA", 0.05),
                       methods=_preT_demo_methods).to_string(index=False))
else:
    print("Skipped PreT sanity run: no method in RESULTS has SSIM values.")

In [ ]:
# ============================================================
# Cell 45 (NEW) - Paper's Experiment 4/5: Efficiency vs. Quality
#   ADDITIVE ONLY: reads RESULTS (post eta-backfill, Cell 42 above) and
#   PALETTE-style colors; writes only new figures/tables, no globals mutated.
#   Y axis: Dictionary Efficiency (eta, %). X axis: SSIM.
#   One curve per method, points/markers at each swept k (K_SWEEP),
#   mean over nodes -- the paper's "efficiency enhancement" curve shape:
#   as k grows, SSIM rises but eta (efficiency per nonzero coefficient)
#   typically falls, so these curves trace out each method's
#   efficiency/quality trade-off.
#
#   UPDATE (this revision): this cell previously hardcoded
#   _eff_methods to OMP/SIBS1-4 only, even though the eta-backfill cell
#   above computes eta for EVERY method family, including "federated"
#   (F-SIBS_1..4) -- so F-SIBS was silently absent from this experiment's
#   only figure. F-SIBS is now appended (own color per variant, clearly
#   distinguished from the local baselines), and two more outputs are
#   added: an overall mean-eta bar chart (all methods on one comparable
#   axis, since 9 line-per-method panels get visually crowded) and a
#   CSV table with the exact per-method summary statistics.
# ============================================================
_EFF_COLORS = {"OMP": "#2c2c2c", "SIBS1": "#1a6eb5", "SIBS2": "#3ba3d9",
              "SIBS3": "#80cbe3", "SIBS4": "#5a8fb8",
              "F-SIBS_1": "#b22222", "F-SIBS_2": "#e06666", "F-SIBS_3": "#e0a020",
              "F-SIBS_4": "#c99a06"}
_eff_methods = [m for m in ["OMP", "SIBS1", "SIBS2", "SIBS3", "SIBS4",
                            "F-SIBS_1", "F-SIBS_2", "F-SIBS_3", "F-SIBS_4"]
               if m in RESULTS["method"].unique()]

if not _eff_methods:
    print("Skipped: Efficiency-vs-Quality plot -- none of OMP/SIBS1-4/F-SIBS_1-4 are "
          "present in RESULTS (check SELECTED_METHODS in the GLOBAL EXPERIMENT "
          "SELECTION cell).")
elif "eta" not in RESULTS.columns or RESULTS["eta"].notna().sum() == 0:
    print("Skipped: Efficiency-vs-Quality plot -- RESULTS has no finite 'eta' values "
          "(run the eta-backfill cell above first).")
else:
    _eff_datasets = [d for d in RESULTS["dataset"].unique()
                     if "SELECTED_DATASETS_SET" not in dir() or d in SELECTED_DATASETS_SET]
    if not _eff_datasets:
        _eff_datasets = list(RESULTS["dataset"].unique())

    fig, axes = plt.subplots(1, len(_eff_datasets), figsize=(6.2 * len(_eff_datasets), 4.8),
                             constrained_layout=True, squeeze=False)
    axes = axes[0]

    for ax, ds in zip(axes, _eff_datasets):
        sub = RESULTS[(RESULTS["dataset"] == ds) & (RESULTS["method"].isin(_eff_methods))
                      & RESULTS["eta"].notna() & RESULTS["SSIM"].notna()]
        any_plotted = False
        for m in _eff_methods:
            msub = (sub[sub["method"] == m]
                   .groupby("k", as_index=False)[["SSIM", "eta"]].mean()
                   .sort_values("SSIM"))
            if msub.empty:
                continue
            _is_federated = m.startswith("F-SIBS")
            ax.plot(msub["SSIM"], msub["eta"],
                    marker="o", linestyle="-" if _is_federated else "--",
                    linewidth=2.2 if _is_federated else 1.4,
                    label=m, color=_EFF_COLORS.get(m, "#888888"))
            any_plotted = True
        ax.set_xlabel("SSIM")
        ax.set_ylabel(r"Dictionary Efficiency $\eta$ (%)")
        ax.set_title("%s — efficiency vs. quality" % ds)
        ax.grid(alpha=0.3)
        if any_plotted:
            ax.legend(fontsize=7, ncol=2)
        else:
            ax.text(0.5, 0.5, "no data", ha="center", va="center", transform=ax.transAxes)

    _eff_fig_path = os.path.join(FIG_DIR, "sibs_exp4_5_efficiency_vs_quality.png")
    plt.savefig(_eff_fig_path)
    plt.show()
    print("Saved:", _eff_fig_path)

    # ------------------------------------------------------------
    # NEW TABLE - per-method summary statistics (mean/std SSIM, eta, PSNR,
    # averaged over every node/dataset/k in this run) for every method in
    # _eff_methods, including F-SIBS_1..4.
    # ------------------------------------------------------------
    _eff_metric_cols = [c for c in ["SSIM", "eta", "PSNR"] if c in RESULTS.columns]
    _eff_summary_rows = []
    for m in _eff_methods:
        _msub = RESULTS[(RESULTS["method"] == m) & RESULTS["eta"].notna()]
        if _msub.empty:
            continue
        _row = {"method": m, "family": "federated" if m.startswith("F-SIBS") else "single",
               "n_rows": len(_msub)}
        for _col in _eff_metric_cols:
            _row["mean_%s" % _col] = float(_msub[_col].mean())
            _row["std_%s" % _col] = float(_msub[_col].std())
        _eff_summary_rows.append(_row)
    EFFICIENCY_SUMMARY_TABLE = pd.DataFrame(_eff_summary_rows).sort_values(
        "mean_eta", ascending=False).reset_index(drop=True)

    print("\nDictionary-efficiency summary per method (mean +/- std over all "
          "nodes/datasets/k in this run):")
    print(EFFICIENCY_SUMMARY_TABLE.round(4).to_string(index=False))
    _eff_table_path = os.path.join(FIG_DIR, "sibs_exp4_5_efficiency_summary_table.csv")
    EFFICIENCY_SUMMARY_TABLE.to_csv(_eff_table_path, index=False)
    print("Saved:", _eff_table_path)

    # ------------------------------------------------------------
    # NEW FIGURE - overall mean-eta bar chart, one bar per method,
    # F-SIBS variants visually grouped/colored apart from the local
    # baselines. Complements the per-dataset line plots above with a
    # single, uncluttered, publication-ready ranking view.
    # ------------------------------------------------------------
    fig2, ax2 = plt.subplots(figsize=(1.1 * len(EFFICIENCY_SUMMARY_TABLE) + 2, 4.8))
    _bar_colors = [_EFF_COLORS.get(m, "#888888") for m in EFFICIENCY_SUMMARY_TABLE["method"]]
    _bars = ax2.bar(EFFICIENCY_SUMMARY_TABLE["method"], EFFICIENCY_SUMMARY_TABLE["mean_eta"],
                    yerr=EFFICIENCY_SUMMARY_TABLE["std_eta"], color=_bar_colors,
                    capsize=4, edgecolor="black", linewidth=0.6)
    for _bar, _fam in zip(_bars, EFFICIENCY_SUMMARY_TABLE["family"]):
        if _fam == "federated":
            _bar.set_hatch("//")
    ax2.set_ylabel(r"Mean Dictionary Efficiency $\eta$ (%)")
    ax2.set_title("Dictionary Efficiency by method (federated variants hatched)")
    ax2.set_xticklabels(EFFICIENCY_SUMMARY_TABLE["method"], rotation=35, ha="right")
    ax2.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    _eff_bar_path = os.path.join(FIG_DIR, "sibs_exp4_5_efficiency_bar_chart.png")
    plt.savefig(_eff_bar_path)
    plt.show()
    print("Saved:", _eff_bar_path)

In [ ]:
# ============================================================
# Cell 46 (NEW) - Paper's Experiment 3: PreT heatmap
#   ADDITIVE ONLY: consumes compute_preT() (defined above) and RESULTS;
#   writes only a new figure/PNG.
#   One row per method, one column per metric in {SSIM, PSNR, eta};
#   cell value = PreT(method, metric) in [0, 1] -- the fraction of rival
#   methods this method is significantly better than on that metric.
# ============================================================
try:
    import seaborn as sns
    HAS_SEABORN = True
except ImportError:
    HAS_SEABORN = False
    print("NOTE: seaborn not available -> falling back to a plain matplotlib heatmap.")

_pret_metrics = [m for m in ["SSIM", "PSNR", "eta"] if m in RESULTS.columns]
_pret_methods = sorted(RESULTS["method"].dropna().unique().tolist())

if not _pret_metrics:
    print("Skipped: PreT heatmap -- none of SSIM/PSNR/eta are present in RESULTS.")
elif not _pret_methods:
    print("Skipped: PreT heatmap -- RESULTS has no methods.")
else:
    _pret_wide = pd.DataFrame(index=_pret_methods, columns=_pret_metrics, dtype=float)
    for metric in _pret_metrics:
        _summary = compute_preT(RESULTS, metric=metric,
                                alpha=globals().get("ALPHA", 0.05), methods=_pret_methods)
        for _, r in _summary.iterrows():
            _pret_wide.loc[r["method"], metric] = r["PreT"]

    pret_csv = os.path.join(FIG_DIR, "sibs_exp3_preT_heatmap.csv")
    _pret_wide.to_csv(pret_csv)

    # Sort rows by mean PreT across metrics (paper's overall ranking order)
    _pret_wide = _pret_wide.loc[_pret_wide.mean(axis=1).sort_values(ascending=False).index]

    fig, ax = plt.subplots(figsize=(1.6 + 1.4 * len(_pret_metrics), 0.45 * len(_pret_methods) + 1.5))
    if HAS_SEABORN:
        sns.heatmap(_pret_wide, annot=True, fmt=".2f", vmin=0.0, vmax=1.0, cmap="viridis",
                   cbar_kws={"label": "PreT"}, linewidths=0.5, linecolor="white", ax=ax)
    else:
        im = ax.imshow(_pret_wide.values.astype(float), vmin=0.0, vmax=1.0, cmap="viridis",
                       aspect="auto")
        ax.set_xticks(range(len(_pret_metrics))); ax.set_xticklabels(_pret_metrics)
        ax.set_yticks(range(len(_pret_methods))); ax.set_yticklabels(_pret_wide.index)
        for i in range(_pret_wide.shape[0]):
            for j in range(_pret_wide.shape[1]):
                v = _pret_wide.values[i, j]
                if np.isfinite(v):
                    ax.text(j, i, "%.2f" % v, ha="center", va="center",
                           color="white" if v < 0.5 else "black", fontsize=8)
        plt.colorbar(im, ax=ax, label="PreT")

    ax.set_title("SIBS-paper Preference Test (PreT) — statistical ranking summary\n"
                "(fraction of rival methods each row is significantly better than)")
    plt.tight_layout()
    plt.savefig(os.path.join(FIG_DIR, "sibs_exp3_preT_heatmap.png"))
    plt.show()
    print("Saved:", os.path.join(FIG_DIR, "sibs_exp3_preT_heatmap.png"), "and", pret_csv)

# ============================================================
# Cell 48 (NEW) - Completing Experiment 3: explicit SigT matrix
#   ADDITIVE ONLY. compute_preT (Cell 43/§11) already computes a pairwise
#   binary significance matrix internally (`sig`) but only ever returns
#   the AGGREGATED PreT score per method - the paper also asks for the
#   SigT matrix itself ("SigT matrix with binary significance outcomes").
#   compute_sigT_matrix() below reuses the EXACT SAME Welch t-test
#   definition as compute_preT (same alpha, same "significant AND
#   mean(A_j) > mean(A_l)" rule) so the two are always consistent, and
#   is purely additive: compute_preT/its call sites are untouched.
# ============================================================

if "RESULTS" in globals() and len(RESULTS) > 0 and "method" in RESULTS.columns:
    _sigt_methods = sorted(RESULTS["method"].dropna().unique().tolist())
    SIGT_SSIM = compute_sigT_matrix(RESULTS, metric="SSIM",
                                    alpha=globals().get("ALPHA", 0.05),
                                    methods=_sigt_methods)
    _sigt_path = save_table(SIGT_SSIM.reset_index().rename(columns={"index": "method"}),
                            3, "sibs_exp3_sigT_matrix_ssim")

    fig, ax = plt.subplots(figsize=(1.2 + 0.55 * len(_sigt_methods),
                                    1.2 + 0.45 * len(_sigt_methods)))
    im = ax.imshow(SIGT_SSIM.values.astype(float), vmin=0, vmax=1, cmap="RdYlGn")
    ax.set_xticks(range(len(_sigt_methods))); ax.set_xticklabels(_sigt_methods, rotation=90, fontsize=7)
    ax.set_yticks(range(len(_sigt_methods))); ax.set_yticklabels(_sigt_methods, fontsize=7)
    ax.set_title("SigT(row significantly beats column) on SSIM, alpha=%.2f"
                % globals().get("ALPHA", 0.05), fontsize=10)
    plt.colorbar(im, ax=ax, label="SigT (1 = row significantly better)")
    plt.tight_layout()
    _sigt_fig = save_figure(fig, 3, "sibs_exp3_sigT_matrix_ssim")
    plt.show()

    EXPERIMENT_MANIFEST[3]["tables"].append("sibs_exp3_sigT_matrix_ssim.csv")
    EXPERIMENT_MANIFEST[3]["figures"].append("sibs_exp3_sigT_matrix_ssim.png")
    print("Saved SigT matrix (SSIM):", _sigt_path)
else:
    print("Skipped: SigT matrix -- RESULTS is empty or has no 'method' column.")

# ---- E3 manifest self-registration ----
for _f in ["sibs_exp3_preT_heatmap.csv", "sibs_exp4_5_efficiency_summary_table.csv",
           "sibs_exp3_sigT_matrix_ssim.csv"]:
    if os.path.exists(os.path.join(FIG_DIR, _f)) and _f not in EXPERIMENT_MANIFEST[3]["tables"]:
        EXPERIMENT_MANIFEST[3]["tables"].append(_f)
for _f in ["sibs_exp3_preT_heatmap.png", "sibs_exp4_5_efficiency_vs_quality.png",
           "sibs_exp4_5_efficiency_bar_chart.png", "sibs_exp3_sigT_matrix_ssim.png"]:
    if os.path.exists(os.path.join(FIG_DIR, _f)) and _f not in EXPERIMENT_MANIFEST[3]["figures"]:
        EXPERIMENT_MANIFEST[3]["figures"].append(_f)


# ## Experiment E4 — Practical Compressor vs. JPEG / JPEG2000
# 
# **Objective.** Evaluate F-SIBS as a practical distributed image compressor under true
# entropy coding, against JPEG / JPEG2000.
# 
# **Inputs:** `POOLS`/`PHI`/`GROUPS`/`CONV_SPECS` (Cell 3), dynamic-SSIM machinery (Cell 2),
# `DYNAMIC_SSIM_*` knobs (Cell 1). **Outputs:** comparison-suite results (fixed-k and
# dynamic-SSIM stopping, raw vs. entropy-coded), federated R-D completion over the augmented
# dictionary `[Phi, global_dict]`, the multi-dataset / multi-N / multi-tau / multi-run
# extension, the **V3.2 Bjøntegaard BD-rate table vs. JPEG & JPEG2000** and the stacked
# bit-budget decomposition (coefficient vs. federation uplink).
# 
# **Figures:** comparison R-D curves, federated R-D curve, extension small-multiples, BD-rate
# bars, bit-budget bars. **Tables:** `exp4_federated_rd_table.csv`,
# `exp4_bd_rate_vs_jpeg_jpeg2000.csv`, `exp4_ext_summary_by_dataset_N_tau.csv`, …
# **Contribution supported:** primary practical-compression result.

## Register outputs

In [ ]:
save_manifest(EXPERIMENT_MANIFEST, "E03")   # lets the F2 report notebook map this notebook's outputs

In [ ]:
# ============================================================
# ENHANCEMENT HELPERS -- shared inline setup cell.
# Implements the "Generate -> Save PNG -> Display inline -> Register
# -> Add to .tex" workflow. All figures are saved as PNG @ 300 DPI to a
# single shared RUN_DIR/figures/ directory (no experiment-specific
# subfolders). The .tex file lives at RUN_DIR/Exp0X_Results.tex.
# ============================================================
import os, sys, json, time, platform
import shutil
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from IPython.display import display, Markdown, Image

# ---- Resolve the shared figures directory and the consolidated .tex ----
# RUN_DIR is set by the original notebook's setup cell. The shared
# figures/ folder lives directly under RUN_DIR (NOT under FIG_DIR, so
# all experiments share one folder). The .tex file also lives directly
# under RUN_DIR.
_RUN_DIR = Path(globals().get("RUN_DIR", ".")).resolve()
SHARED_FIG_DIR = _RUN_DIR / "figures"
SHARED_FIG_DIR.mkdir(parents=True, exist_ok=True)

def _escape_latex(s):
    """Escape LaTeX special characters in a string for safe inclusion in
    table cells and captions."""
    if s is None:
        return ""
    s = str(s)
    for ch, repl in [("\\", r"\textbackslash{}"), ("&", r"\&"),
                     ("%", r"\%"), ("$", r"\$"), ("#", r"\#"),
                     ("_", r"\_"), ("{", r"\{"), ("}", r"\}"),
                     ("~", r"\textasciitilde{}"), ("^", r"\textasciicircum{}")]:
        s = s.replace(ch, repl)
    return s

def _fmt_num(v, nd=4):
    """Format a numeric value for LaTeX, handling NaN gracefully."""
    if v is None:
        return "---"
    try:
        f = float(v)
    except (TypeError, ValueError):
        return _escape_latex(v)
    if not np.isfinite(f):
        return "---"
    if isinstance(v, (int, np.integer)) and float(v) == f:
        return str(int(v))
    return f"{f:.{nd}f}"

# ---- State: the consolidated .tex file path and the open buffer -------
_EXPERIMENT_TEX_PATH = None      # set by init_experiment_tex
_EXPERIMENT_TEX_LINES = []       # list of str lines accumulated so far
_FIGURE_REGISTRY = {}            # id -> {filename, label, caption, purpose, ...}
_TABLE_REGISTRY = {}              # id -> {label, caption, purpose, ...}

def init_experiment_tex(experiment_id, title, intro_latex, notebook_filename):
    """Initialize (or reset) the consolidated experiment .tex file.

    Saves the path and writes the file header + intro to the in-memory
    buffer. The file is written to disk by finalize_experiment_tex()
    OR by save_experiment_tex_now() (which can be called between cells
    to flush a partial file)."""
    global _EXPERIMENT_TEX_PATH, _EXPERIMENT_TEX_LINES
    _EXPERIMENT_TEX_PATH = _RUN_DIR / f"Exp{experiment_id}_Results.tex"
    _EXPERIMENT_TEX_LINES = []
    _EXPERIMENT_TEX_LINES.append(r"% ============================================================================")
    _EXPERIMENT_TEX_LINES.append(r"% Exp" + str(experiment_id) + r"_Results.tex")
    _EXPERIMENT_TEX_LINES.append(r"% Consolidated LaTeX results file for Experiment " + str(experiment_id) + ".")
    _EXPERIMENT_TEX_LINES.append(r"% AUTO-GENERATED BY " + str(notebook_filename) + ".")
    _EXPERIMENT_TEX_LINES.append(r"% Numerical values come from the live in-memory DataFrames.")
    _EXPERIMENT_TEX_LINES.append(r"% Do not edit by hand; re-run the notebook to regenerate.")
    _EXPERIMENT_TEX_LINES.append(r"% Figures are PNG @ 300 DPI, saved to the shared RUN_DIR/figures/ folder.")
    _EXPERIMENT_TEX_LINES.append(r"% ============================================================================")
    _EXPERIMENT_TEX_LINES.append("")
    _EXPERIMENT_TEX_LINES.append(r"% --- Experiment " + str(experiment_id) + " introduction ---")
    _EXPERIMENT_TEX_LINES.append(r"\section*{" + _escape_latex(title) + "}")
    _EXPERIMENT_TEX_LINES.append("")
    if intro_latex:
        _EXPERIMENT_TEX_LINES.append(intro_latex.strip("\n"))
        _EXPERIMENT_TEX_LINES.append("")
    print(f"Initialized: {_EXPERIMENT_TEX_PATH}")
    print(f"Shared figures directory: {SHARED_FIG_DIR}")
    save_experiment_tex_now()
    return _EXPERIMENT_TEX_PATH

def save_experiment_tex_now():
    """Flush the current in-memory .tex buffer to disk."""
    if _EXPERIMENT_TEX_PATH is None:
        return None
    with open(_EXPERIMENT_TEX_PATH, "w") as f:
        f.write("\n".join(_EXPERIMENT_TEX_LINES) + "\n")
    return _EXPERIMENT_TEX_PATH

def finalize_experiment_tex(experiment_id, observations_latex=None,
                            reproducibility_dict=None):
    """Append observations, figure/table registry, and reproducibility
    metadata as LaTeX comments, then flush to disk."""
    if observations_latex:
        _EXPERIMENT_TEX_LINES.append(r"% --- Scientific observations ---")
        _EXPERIMENT_TEX_LINES.append(observations_latex.strip("\n"))
        _EXPERIMENT_TEX_LINES.append("")
    _EXPERIMENT_TEX_LINES.append(r"% ===========================================================================")
    _EXPERIMENT_TEX_LINES.append(r"% Figure & Table Registry (machine-readable, for AI paper-writing agents)")
    _EXPERIMENT_TEX_LINES.append(r"% ===========================================================================")
    _EXPERIMENT_TEX_LINES.append(r"% Figures:")
    for fig_id, info in _FIGURE_REGISTRY.items():
        _EXPERIMENT_TEX_LINES.append(
            r"% " + fig_id + " | file: " + info["filename"]
            + r" | label: " + info["label"])
        _EXPERIMENT_TEX_LINES.append(r"%   purpose: " + info.get("purpose", ""))
        _EXPERIMENT_TEX_LINES.append(
            r"%   dataset: " + info.get("dataset", "")
            + r" | methods: " + info.get("methods", "")
            + r" | metric: " + info.get("metrics", ""))
        _EXPERIMENT_TEX_LINES.append(
            r"%   finding: " + info.get("scientific_finding", ""))
    _EXPERIMENT_TEX_LINES.append(r"% Tables:")
    for tbl_id, info in _TABLE_REGISTRY.items():
        _EXPERIMENT_TEX_LINES.append(
            r"% " + tbl_id + r" | label: " + info["label"])
        _EXPERIMENT_TEX_LINES.append(r"%   purpose: " + info.get("purpose", ""))
        _EXPERIMENT_TEX_LINES.append(
            r"%   dataset: " + info.get("dataset", "")
            + r" | methods: " + info.get("methods", "")
            + r" | metrics: " + info.get("metrics", ""))
    if reproducibility_dict:
        _EXPERIMENT_TEX_LINES.append(r"% ===========================================================================")
        _EXPERIMENT_TEX_LINES.append(r"% Reproducibility metadata")
        _EXPERIMENT_TEX_LINES.append(r"% ===========================================================================")
        for k, v in reproducibility_dict.items():
            if isinstance(v, (dict, list)):
                _EXPERIMENT_TEX_LINES.append(r"% " + k + ": " + json.dumps(v, default=str))
            else:
                _EXPERIMENT_TEX_LINES.append(r"% " + k + ": " + str(v))
    save_experiment_tex_now()
    print(f"Finalized: {_EXPERIMENT_TEX_PATH}")
    print(f"  ({len(_EXPERIMENT_TEX_LINES)} lines, "
          f"{len(_FIGURE_REGISTRY)} figures, "
          f"{len(_TABLE_REGISTRY)} tables registered)")
    return _EXPERIMENT_TEX_PATH

def save_and_display_fig(fig, fig_id, filename, caption, label, purpose,
                         dataset="", methods="", metrics="",
                         scientific_finding="", dpi=300,
                         bbox_inches="tight", close_after=True):
    """Generate -> Save PNG @ DPI -> Display inline -> Register -> Append
    \\includegraphics block to the consolidated .tex.

    The figure is saved to SHARED_FIG_DIR/<filename> (PNG, no other
    format). The LaTeX \\includegraphics references
    `figures/<filename>` (with the .png extension).
    """
    # Save PNG @ DPI=300 with tight bbox (only if scientifically appropriate
    # -- here we always use bbox_inches="tight" since the figures are
    # publication bar/line charts where tight is the right choice)
    out_path = SHARED_FIG_DIR / filename
    fig.savefig(out_path, dpi=dpi, bbox_inches=bbox_inches,
                facecolor=fig.get_facecolor() if hasattr(fig, "get_facecolor") else "white")
    # Display inline immediately
    display(fig)
    if close_after:
        plt.close(fig)
    # Register
    _FIGURE_REGISTRY[fig_id] = {
        "id": fig_id, "filename": filename, "label": label,
        "caption": caption, "purpose": purpose,
        "dataset": dataset, "methods": methods, "metrics": metrics,
        "scientific_finding": scientific_finding,
    }
    # Append the LaTeX figure block to the consolidated .tex
    _EXPERIMENT_TEX_LINES.append(r"% Figure " + fig_id)
    _EXPERIMENT_TEX_LINES.append(r"\begin{figure}[t]")
    _EXPERIMENT_TEX_LINES.append(r"  \centering")
    _EXPERIMENT_TEX_LINES.append(
        r"  \includegraphics[width=\linewidth]{figures/" + filename + "}")
    _EXPERIMENT_TEX_LINES.append(r"  \caption{" + _escape_latex(caption) + "}")
    _EXPERIMENT_TEX_LINES.append(r"  \label{" + label + "}")
    _EXPERIMENT_TEX_LINES.append(r"\end{figure}")
    _EXPERIMENT_TEX_LINES.append("")
    # Print a brief summary line so the cell's stdout is useful too
    print(f"  {fig_id}: saved {filename} ({dpi} DPI) -> displayed inline -> added to .tex")
    return out_path

def display_and_export_table(df, tbl_id, caption, label, purpose,
                             dataset="", methods="", metrics="",
                             scientific_finding="", ncols_spec=None,
                             max_rows=None, fontsize=r"\small",
                             show_index=False):
    """Display a DataFrame inline immediately, then generate its LaTeX
    representation and append it to the consolidated .tex.

    The displayed DataFrame and the LaTeX table are guaranteed to contain
    the same numerical values (the LaTeX is generated directly from the
    same df via pandas .to_latex()).
    """
    # Optionally limit rows for the LaTeX table (publication-feasibility)
    df_display = df
    df_latex = df
    if max_rows is not None and len(df) > max_rows:
        df_latex = df.head(max_rows)
        _truncation_note = (f"  ({len(df) - max_rows} additional rows "
                             "omitted for brevity in the LaTeX table)")
    else:
        _truncation_note = ""
    # 1. Display inline immediately
    display(df_display)
    # 2. Generate LaTeX representation
    try:
        if ncols_spec is None:
            ncols_spec = "l" + "r" * (len(df_latex.columns) - 1) if len(df_latex.columns) > 0 else "l"
        latex_body = df_latex.to_latex(index=show_index, escape=False,
                                       column_format=ncols_spec, na_rep="---")
        # Wrap with the table environment + caption + label
        latex_table = []
        latex_table.append(r"\begin{table}[t]")
        latex_table.append(r"\centering")
        latex_table.append(r"\caption{" + _escape_latex(caption) + "}")
        latex_table.append(r"\label{" + label + "}")
        latex_table.append(fontsize)
        latex_table.append(latex_body.rstrip())
        if _truncation_note:
            latex_table.append(r"\multicolumn{" + str(len(df_latex.columns)) +
                               r"}{l}{\textit{" + _truncation_note.strip() + r"}}")
        latex_table.append(r"\end{table}")
        latex_table_str = "\n".join(latex_table)
    except Exception as e:
        latex_table_str = f"% LaTeX table generation failed: {e!r}"
        print("  WARNING:", latex_table_str)
    # 3. Register the table
    _TABLE_REGISTRY[tbl_id] = {
        "id": tbl_id, "label": label, "caption": caption,
        "purpose": purpose, "dataset": dataset, "methods": methods,
        "metrics": metrics, "scientific_finding": scientific_finding,
    }
    # 4. Append to the consolidated .tex
    _EXPERIMENT_TEX_LINES.append(r"% Table " + tbl_id)
    _EXPERIMENT_TEX_LINES.append(latex_table_str)
    _EXPERIMENT_TEX_LINES.append("")
    print(f"  {tbl_id}: displayed inline ({len(df_display)} rows) -> LaTeX added to .tex")
    return latex_table_str

def capture_reproducibility(experiment_id, notebook_filename):
    """Capture Python version, platform, CPU, package versions, run folder,
    and experiment configuration. Returns a dict; caller is responsible
    for displaying it inline and/or passing to finalize_experiment_tex()."""
    try:
        import importlib.metadata as _md
    except ImportError:
        import importlib_metadata as _md  # type: ignore
    packages = {}
    for _pkg in ["numpy", "scipy", "pandas", "matplotlib", "scikit-image",
                 "tqdm", "seaborn", "fsibs", "imagecodecs"]:
        try:
            packages[_pkg] = _md.version(_pkg)
        except Exception:
            packages[_pkg] = "NOT INSTALLED"
    repro = {
        "experiment_id": str(experiment_id),
        "notebook_file": str(notebook_filename),
        "python_version": sys.version,
        "platform": platform.platform(),
        "cpu_count": os.cpu_count(),
        "execution_timestamp": time.strftime("%Y-%m-%d %H:%M:%S %z"),
        "packages": packages,
        "run_folder": str(_RUN_DIR),
        "shared_figures_dir": str(SHARED_FIG_DIR),
        "experiment_tex_path": (str(_EXPERIMENT_TEX_PATH)
                                if _EXPERIMENT_TEX_PATH else None),
        "experiment_configuration": {
            "SELECTED_DATASETS": list(globals().get("SELECTED_DATASETS", [])),
            "SELECTED_ALGORITHMS": list(globals().get("SELECTED_ALGORITHMS", [])),
            "K_SUMMARY": globals().get("K_SUMMARY", "[REQUIRES CONFIRMATION]"),
            "K_GLOBAL": globals().get("K_GLOBAL", "[REQUIRES CONFIRMATION]"),
            "DELTA": globals().get("DELTA", "[REQUIRES CONFIRMATION]"),
            "N_ROUNDS_SWEEP": globals().get("N_ROUNDS_SWEEP", "[REQUIRES CONFIRMATION]"),
            "EPS_CONV": globals().get("EPS_CONV", "[REQUIRES CONFIRMATION]"),
            "RNG_SEED": globals().get("RNG_SEED", "[REQUIRES CONFIRMATION]"),
            "RUN_DIR_POLICY": globals().get("RUN_DIR_POLICY", "latest"),
            "INTERACTIVE_SELECTION": globals().get("INTERACTIVE_SELECTION", False),
        },
    }
    return repro

def display_reproducibility(repro_dict):
    """Display the reproducibility metadata inline as a small DataFrame."""
    rows = []
    rows.append({"field": "experiment_id", "value": repro_dict["experiment_id"]})
    rows.append({"field": "notebook_file", "value": repro_dict["notebook_file"]})
    rows.append({"field": "python_version", "value": repro_dict["python_version"]})
    rows.append({"field": "platform", "value": repro_dict["platform"]})
    rows.append({"field": "cpu_count", "value": repro_dict["cpu_count"]})
    rows.append({"field": "execution_timestamp", "value": repro_dict["execution_timestamp"]})
    rows.append({"field": "run_folder", "value": repro_dict["run_folder"]})
    rows.append({"field": "shared_figures_dir", "value": repro_dict["shared_figures_dir"]})
    rows.append({"field": "experiment_tex_path", "value": repro_dict["experiment_tex_path"]})
    rows.append({"field": "experiment_configuration", "value": json.dumps(repro_dict["experiment_configuration"], default=str, indent=2)})
    rows.append({"field": "packages", "value": json.dumps(repro_dict["packages"], default=str, indent=2)})
    display(pd.DataFrame(rows))

def copy_and_display_existing_png(src_path, fig_id, filename, caption, label,
                                  purpose, dataset="", methods="",
                                  metrics="", scientific_finding=""):
    """For figures that are too expensive to recompute (e.g. a
    reconstruction gallery that requires running all algorithms), copy
    the existing PNG saved by the original cell to the shared figures/
    folder with the new paper-friendly filename, then display inline.

    This is a fallback for figures where regeneration from a DataFrame
    is not practical. The preferred path remains save_and_display_fig()
    with a freshly-generated matplotlib figure at DPI=300.
    """
    src = Path(src_path)
    dst = SHARED_FIG_DIR / filename
    if not src.exists():
        print(f"  WARNING: source PNG not found: {src}")
        print(f"           Run the original experiment cell that produces it first.")
        return None
    shutil.copyfile(src, dst)
    # Display inline immediately via IPython.display.Image (so the
    # notebook shows the actual PNG, not a matplotlib figure)
    display(Image(filename=str(dst)))
    # Register
    _FIGURE_REGISTRY[fig_id] = {
        "id": fig_id, "filename": filename, "label": label,
        "caption": caption, "purpose": purpose,
        "dataset": dataset, "methods": methods, "metrics": metrics,
        "scientific_finding": scientific_finding,
    }
    # Append the LaTeX figure block to the consolidated .tex
    _EXPERIMENT_TEX_LINES.append(r"% Figure " + fig_id + " (copied from original cell's PNG)")
    _EXPERIMENT_TEX_LINES.append(r"\begin{figure}[t]")
    _EXPERIMENT_TEX_LINES.append(r"  \centering")
    _EXPERIMENT_TEX_LINES.append(
        r"  \includegraphics[width=\linewidth]{figures/" + filename + "}")
    _EXPERIMENT_TEX_LINES.append(r"  \caption{" + _escape_latex(caption) + "}")
    _EXPERIMENT_TEX_LINES.append(r"  \label{" + label + "}")
    _EXPERIMENT_TEX_LINES.append(r"\end{figure}")
    _EXPERIMENT_TEX_LINES.append("")
    print(f"  {fig_id}: copied {src.name} -> {filename} -> displayed inline -> added to .tex")
    return dst


# ============================================================
# E03 — Capture reproducibility + initialize Exp03_Results.tex
# ============================================================
_EXPERIMENT_ID = "03"
_NOTEBOOK_FILE = "E03_dictionary_efficiency_preT_sigT.ipynb"
_TITLE = "Experiment E03 -- Dictionary Efficiency ($\\eta$), PreT and SigT"
_INTRO_LATEX = r"""\noindent This experiment analyses the main-sweep results to establish
the dictionary-efficiency and statistical-dominance arguments for the
shared federated dictionary. The dictionary efficiency
$\eta = (1 - \text{NMSE}) / \text{NNZC} \cdot 100$ is backfilled into
\texttt{RESULTS} (with $\text{NNZC} = k \cdot w$ for this notebook's
column-wise sparse coding). The SIBS-paper Preference Test (PreT) is
computed over $\{\text{SSIM}, \text{PSNR}, \eta\}$ for every method in
\texttt{RESULTS}, and the explicit pairwise binary significance matrix
(SigT) is computed on SSIM using a Welch independent-samples t-test at
$\alpha = 0.05$."""

init_experiment_tex(_EXPERIMENT_ID, _TITLE, _INTRO_LATEX, _NOTEBOOK_FILE)

_REPRO_E03 = capture_reproducibility(_EXPERIMENT_ID, _NOTEBOOK_FILE)
display_reproducibility(_REPRO_E03)


In [ ]:
# ============================================================
# E03 — FIGURES
# ============================================================
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

_required = ["RESULTS", "EFFICIENCY_SUMMARY_TABLE"]
_missing = [n for n in _required if n not in globals()]
if _missing:
    print("Skipped: E03 figures. Missing required state: " + ", ".join(_missing))
else:
    K_SUMMARY = globals().get("K_SUMMARY", 8)
    K_SWEEP = globals().get("K_SWEEP", [])
    POOLS = globals().get("POOLS", {})
    SELECTED_DATASETS_SET = globals().get("SELECTED_DATASETS_SET", set())
    _EFF_COLORS = {"OMP": "#2c2c2c", "SIBS1": "#1a6eb5", "SIBS2": "#3ba3d9",
                    "SIBS3": "#80cbe3", "SIBS4": "#5a8fb8",
                    "F-SIBS_1": "#b22222", "F-SIBS_2": "#e06666",
                    "F-SIBS_3": "#e0a020", "F-SIBS_4": "#c99a06"}
    _eff_methods = [m for m in ["OMP", "SIBS1", "SIBS2", "SIBS3", "SIBS4",
                                "F-SIBS_1", "F-SIBS_2", "F-SIBS_3", "F-SIBS_4"]
                    if m in RESULTS["method"].unique()]

    # ===== EXP03-F01: PreT heat-map =================================
    if "_pret_wide" not in globals() or len(_pret_wide) == 0:
        print("Skipped: EXP03-F01 (PreT heat-map) -- _pret_wide not in scope.")
    else:
        try:
            import seaborn as sns
            _has_seaborn = True
        except ImportError:
            _has_seaborn = False
        _pret = _pret_wide.copy()
        _pret = _pret.loc[_pret.mean(axis=1).sort_values(ascending=False).index]
        fig, ax = plt.subplots(figsize=(1.6 + 1.4 * len(_pret.columns),
                                         0.45 * len(_pret) + 1.5),
                                constrained_layout=True)
        if _has_seaborn:
            sns.heatmap(_pret, annot=True, fmt=".2f", vmin=0.0, vmax=1.0,
                        cmap="viridis", cbar_kws={"label": "PreT"},
                        linewidths=0.5, linecolor="white", ax=ax)
        else:
            _im = ax.imshow(_pret.values.astype(float), vmin=0.0, vmax=1.0,
                            cmap="viridis", aspect="auto")
            ax.set_xticks(range(len(_pret.columns)))
            ax.set_xticklabels(_pret.columns)
            ax.set_yticks(range(len(_pret)))
            ax.set_yticklabels(_pret.index)
            for i in range(_pret.shape[0]):
                for j in range(_pret.shape[1]):
                    v = _pret.values[i, j]
                    if np.isfinite(v):
                        ax.text(j, i, f"{v:.2f}", ha="center", va="center",
                                color="white" if v < 0.5 else "black", fontsize=8)
            fig.colorbar(_im, ax=ax, label="PreT")
        ax.set_title("SIBS-paper Preference Test (PreT)\n(fraction of rival methods each row is significantly better than)")
        save_and_display_fig(
            fig, fig_id="EXP03-F01", filename="exp03_pret_heatmap.png",
            caption="SIBS-paper Preference Test (PreT) heat-map. Each cell is the fraction of rival methods that the row method is significantly better than on the column metric (Welch t-test, alpha = 0.05). Rows are sorted by mean PreT across metrics.",
            label="fig:exp03-pret-heatmap",
            purpose="Statistical ranking summary: which methods dominate which on which metrics.",
            dataset="all selected datasets (pooled)",
            methods="All methods present in RESULTS",
            metrics="SSIM, PSNR, eta (where available)",
            scientific_finding="[filled in observations cell]",
        )

    # ===== EXP03-F02: efficiency vs. quality (eta vs SSIM Pareto) ====
    if "eta" not in RESULTS.columns or RESULTS["eta"].notna().sum() == 0:
        print("Skipped: EXP03-F02 (efficiency vs quality) -- RESULTS has no finite eta.")
    else:
        _eff_datasets = [d for d in RESULTS["dataset"].unique()
                         if d in SELECTED_DATASETS_SET] or list(RESULTS["dataset"].unique())
        fig, axes = plt.subplots(1, len(_eff_datasets),
                                  figsize=(6.2 * len(_eff_datasets), 4.8),
                                  constrained_layout=True, squeeze=False)
        axes = axes[0]
        for ax, ds in zip(axes, _eff_datasets):
            sub = RESULTS[(RESULTS["dataset"] == ds) & (RESULTS["method"].isin(_eff_methods))
                           & RESULTS["eta"].notna() & RESULTS["SSIM"].notna()]
            any_plotted = False
            for m in _eff_methods:
                msub = (sub[sub["method"] == m]
                        .groupby("k", as_index=False)[["SSIM", "eta"]].mean()
                        .sort_values("SSIM"))
                if msub.empty: continue
                _is_fed = m.startswith("F-SIBS")
                ax.plot(msub["SSIM"], msub["eta"], marker="o",
                        linestyle="-" if _is_fed else "--",
                        linewidth=2.2 if _is_fed else 1.4,
                        label=m, color=_EFF_COLORS.get(m, "#888888"))
                any_plotted = True
            ax.set_xlabel("SSIM")
            ax.set_ylabel(r"Dictionary Efficiency $\eta$ (%)")
            ax.set_title(f"{ds} -- efficiency vs. quality")
            ax.grid(alpha=0.3)
            if any_plotted: ax.legend(fontsize=7, ncol=2)
            else: ax.text(0.5, 0.5, "no data", ha="center", va="center",
                          transform=ax.transAxes)
        save_and_display_fig(
            fig, fig_id="EXP03-F02",
            filename="exp03_efficiency_vs_quality.png",
            caption="Dictionary efficiency eta vs. quality (SSIM) Pareto scatter, one panel per dataset, one curve per method (markers at each swept k in K_SWEEP). Federated F-SIBS variants are drawn as solid lines; local baselines as dashed lines.",
            label="fig:exp03-efficiency-vs-quality",
            purpose="Headline visualisation of the dictionary-efficiency / quality trade-off: as k grows, SSIM rises but eta typically falls; the federated F-SIBS variants should lie on the Pareto front.",
            dataset="all selected datasets",
            methods="OMP, SIBS1..SIBS4, F-SIBS_1..F-SIBS_4",
            metrics="eta (%, mean over nodes per k) vs. SSIM (mean over nodes per k)",
            scientific_finding="[filled in observations cell]",
        )

    # ===== EXP03-F03: efficiency bar chart ==========================
    fig2, ax2 = plt.subplots(figsize=(1.1 * len(EFFICIENCY_SUMMARY_TABLE) + 2, 4.8),
                              constrained_layout=True)
    _bar_colors = [_EFF_COLORS.get(m, "#888888")
                   for m in EFFICIENCY_SUMMARY_TABLE["method"]]
    _bars = ax2.bar(EFFICIENCY_SUMMARY_TABLE["method"],
                     EFFICIENCY_SUMMARY_TABLE["mean_eta"],
                     yerr=EFFICIENCY_SUMMARY_TABLE["std_eta"],
                     color=_bar_colors, capsize=4,
                     edgecolor="black", linewidth=0.6)
    for _bar, _fam in zip(_bars, EFFICIENCY_SUMMARY_TABLE["family"]):
        if _fam == "federated":
            _bar.set_hatch("//")
    ax2.set_ylabel(r"Mean Dictionary Efficiency $\eta$ (%)")
    ax2.set_title("Dictionary Efficiency by method (federated variants hatched)")
    ax2.set_xticklabels(EFFICIENCY_SUMMARY_TABLE["method"], rotation=35, ha="right")
    ax2.grid(axis="y", alpha=0.3)
    save_and_display_fig(
        fig2, fig_id="EXP03-F03",
        filename="exp03_efficiency_bar_chart.png",
        caption="Mean dictionary efficiency eta per method, with +/- 1 std error bars. Federated F-SIBS variants are hatched to distinguish them visually from the local baselines.",
        label="fig:exp03-efficiency-bar-chart",
        purpose="Single-axis ranking view of the dictionary-efficiency ordering; complements the per-dataset Pareto view of EXP03-F02.",
        dataset="all selected datasets (pooled)",
        methods="OMP, SIBS1..SIBS4, F-SIBS_1..F-SIBS_4",
        metrics="Mean eta (%) with std error bars",
        scientific_finding="[filled in observations cell]",
    )

    # ===== EXP03-F04: SigT matrix on SSIM ===========================
    if "SIGT_SSIM" not in globals() or len(SIGT_SSIM) == 0:
        print("Skipped: EXP03-F04 (SigT matrix) -- SIGT_SSIM not in scope.")
    else:
        _sigt = SIGT_SSIM.copy()
        _methods_sigt = list(_sigt.index)
        fig, ax = plt.subplots(figsize=(1.2 + 0.55 * len(_methods_sigt),
                                         1.2 + 0.45 * len(_methods_sigt)),
                                constrained_layout=True)
        _im = ax.imshow(_sigt.values.astype(float), vmin=0, vmax=1, cmap="RdYlGn")
        ax.set_xticks(range(len(_methods_sigt)))
        ax.set_xticklabels(_methods_sigt, rotation=90, fontsize=7)
        ax.set_yticks(range(len(_methods_sigt)))
        ax.set_yticklabels(_methods_sigt, fontsize=7)
        ax.set_title(f"SigT(row significantly beats column) on SSIM, alpha={globals().get('ALPHA', 0.05)}",
                     fontsize=10)
        fig.colorbar(_im, ax=ax, label="SigT (1 = row significantly better)")
        save_and_display_fig(
            fig, fig_id="EXP03-F04",
            filename="exp03_sigt_matrix_ssim.png",
            caption="Pairwise binary significance matrix SigT(A_j, A_l) on SSIM (Welch t-test, alpha = 0.05). Cell (j, l) = 1 iff A_j significantly beats A_l; 0 otherwise. Diagonal is undefined (NaN). Green = row dominates column.",
            label="fig:exp03-sigt-matrix-ssim",
            purpose="Explicit pairwise dominance structure on SSIM, complementing the aggregated PreT view.",
            dataset="all selected datasets (pooled)",
            methods="All methods present in RESULTS",
            metrics="SSIM (binary significance at alpha = 0.05)",
            scientific_finding="[filled in observations cell]",
        )

save_experiment_tex_now()


In [ ]:
# ============================================================
# E03 — TABLES
# ============================================================
import numpy as np
import pandas as pd
from IPython.display import display

if "EFFICIENCY_SUMMARY_TABLE" not in globals():
    print("Skipped: E03 tables. EFFICIENCY_SUMMARY_TABLE not in scope.")
else:
    # ----- EXP03-T01: EFFICIENCY_SUMMARY_TABLE ----------------------
    _t01 = EFFICIENCY_SUMMARY_TABLE.copy()
    _show = ["method", "family", "n_rows", "mean_SSIM", "std_SSIM",
             "mean_eta", "std_eta", "mean_PSNR", "std_PSNR"]
    _show = [c for c in _show if c in _t01.columns]
    _t01 = _t01[_show].copy()
    for _col in ["mean_SSIM", "std_SSIM", "mean_eta", "std_eta"]:
        if _col in _t01.columns:
            _t01[_col] = _t01[_col].round(4)
    for _col in ["mean_PSNR", "std_PSNR"]:
        if _col in _t01.columns:
            _t01[_col] = _t01[_col].round(3)
    display_and_export_table(
        _t01, tbl_id="EXP03-T01",
        caption="Dictionary-efficiency summary per method (mean +/- std over all nodes / datasets / k in this run). Methods are sorted by mean eta descending.",
        label="tab:exp03-efficiency-summary",
        purpose="Headline numerical reference for the dictionary-efficiency argument.",
        dataset="all selected datasets (pooled)",
        methods="OMP, SIBS1..SIBS4, F-SIBS_1..F-SIBS_4 (only those present in RESULTS)",
        metrics="mean/std SSIM, eta, PSNR; n_rows",
        scientific_finding="[filled in observations cell]",
        ncols_spec="ll" + "r" * (len(_show) - 2),
        fontsize=r"\small",
    )

    # ----- EXP03-T02: _pret_wide (PreT per method x metric) ---------
    if "_pret_wide" not in globals() or len(_pret_wide) == 0:
        print("Skipped: EXP03-T02 (PreT ranking) -- _pret_wide not in scope.")
    else:
        _t02 = _pret_wide.copy().reset_index()
        _t02 = _t02.rename(columns={"index": "method"})
        if "method" not in _t02.columns:
            _t02.columns = ["method"] + list(_t02.columns[1:])
        for _col in _t02.columns:
            if _col != "method" and pd.api.types.is_numeric_dtype(_t02[_col]):
                _t02[_col] = _t02[_col].round(3)
        display_and_export_table(
            _t02, tbl_id="EXP03-T02",
            caption="SIBS-paper Preference Test (PreT) per (method, metric), in [0, 1]. Each cell is the fraction of rival methods that the row method is significantly better than on the column metric.",
            label="tab:exp03-pret-ranking",
            purpose="Numerical reference for the statistical-ranking claim.",
            dataset="all selected datasets (pooled)",
            methods="All methods present in RESULTS",
            metrics="PreT(SSIM), PreT(PSNR), PreT(eta)",
            scientific_finding="[filled in observations cell]",
            ncols_spec="l" + "r" * (len(_t02.columns) - 1),
            fontsize=r"\small",
        )

    # ----- EXP03-T03: SIGT_SSIM (pairwise binary significance) -----
    if "SIGT_SSIM" not in globals() or len(SIGT_SSIM) == 0:
        print("Skipped: EXP03-T03 (SigT matrix) -- SIGT_SSIM not in scope.")
    else:
        _t03 = SIGT_SSIM.copy().reset_index().rename(columns={"index": "method"})
        if "method" not in _t03.columns:
            _t03.columns = ["method"] + list(_t03.columns[1:])
        # Replace NaN diagonal with a sentinel string for display
        _t03_display = _t03.copy()
        for _col in _t03_display.columns:
            if _col != "method":
                _t03_display[_col] = _t03_display[_col].apply(
                    lambda v: int(v) if (pd.notna(v) and float(v) == int(v)) else "--")
        display_and_export_table(
            _t03_display, tbl_id="EXP03-T03",
            caption="Pairwise binary significance matrix SigT(A_j, A_l) on SSIM (Welch t-test, alpha = 0.05). Cell (j, l) = 1 iff A_j is significantly better than A_l on SSIM. Diagonal is undefined (NaN, shown as --).",
            label="tab:exp03-sigt-matrix-ssim",
            purpose="Pairwise dominance structure on SSIM.",
            dataset="all selected datasets (pooled)",
            methods="All methods present in RESULTS",
            metrics="Binary significance on SSIM",
            scientific_finding="[filled in observations cell]",
            ncols_spec="l" + "r" * (len(_t03_display.columns) - 1),
            fontsize=r"\scriptsize",
        )

save_experiment_tex_now()


In [ ]:
# ============================================================
# E03 — OBSERVATIONS + FINALIZE .TEX + AI-AGENT EXPERIMENT RECORD
# ============================================================
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

if "EFFICIENCY_SUMMARY_TABLE" not in globals():
    print("Skipped: E03 finalize. EFFICIENCY_SUMMARY_TABLE not in scope.")
else:
    _n_methods_eff = len(EFFICIENCY_SUMMARY_TABLE)
    _best_eta = EFFICIENCY_SUMMARY_TABLE.iloc[0]
    _fed_methods_eff = EFFICIENCY_SUMMARY_TABLE[EFFICIENCY_SUMMARY_TABLE["family"] == "federated"]
    _best_fed_eta = _fed_methods_eff.iloc[0] if len(_fed_methods_eff) else None

    _obs_latex = []
    _obs_latex.append(r"\paragraph{Observations.}")
    _obs_latex.append(
        r"\noindent Across " + str(_n_methods_eff) + r" methods "
        + r"(Table~\ref{tab:exp03-efficiency-summary}), the highest mean "
        + r"$\eta$ is achieved by "
        + r"\texttt{" + _escape_latex(str(_best_eta["method"])) + r"} ("
        + _fmt_num(_best_eta["mean_eta"], 4) + r"\%)."
    )
    if _best_fed_eta is not None:
        _obs_latex.append(
            r"\noindent Among the federated F-SIBS variants, the best "
            + r"mean $\eta$ is "
            + r"\texttt{" + _escape_latex(str(_best_fed_eta["method"])) + r"} ("
            + _fmt_num(_best_fed_eta["mean_eta"], 4) + r"\%)."
        )
    if "_pret_wide" in globals() and len(_pret_wide) > 0:
        _pret_top = _pret_wide.head(3)
        _obs_latex.append(
            r"\noindent The top-3 methods by mean PreT across metrics "
            + r"(Table~\ref{tab:exp03-pret-ranking}) are: "
        )
        _parts = []
        for method, row in _pret_top.iterrows():
            _mean = float(row.dropna().mean()) if len(row) > 0 else float("nan")
            _parts.append(r"\texttt{" + _escape_latex(str(method)) + r"} ("
                          + _fmt_num(_mean, 3) + r")")
        _obs_latex.append(", ".join(_parts) + r".")
    if "SIGT_SSIM" in globals() and len(SIGT_SSIM) > 0:
        _n_dominators = int((SIGT_SSIM.sum(axis=1) == (SIGT_SSIM.shape[1] - 1)).sum())
        _obs_latex.append(
            r"\noindent On SSIM, " + str(_n_dominators) + r" method(s) "
            + r"dominate every rival at $\alpha = 0.05$ "
            + r"(Table~\ref{tab:exp03-sigt-matrix-ssim})."
        )

    _obs_latex_str = "\n".join(_obs_latex)
    _tex_path = finalize_experiment_tex(_EXPERIMENT_ID,
                                        observations_latex=_obs_latex_str,
                                        reproducibility_dict=_REPRO_E03)

    # ----- AI-Agent Experiment Record (rendered inline) --------------
    try:
        _eff_summary = (f"{_n_methods_eff} methods compared on mean dictionary "
                        f"efficiency (eta); best = `{_best_eta['method']}` "
                        f"(eta = {_best_eta['mean_eta']:.4f}%).")
    except Exception:
        _eff_summary = "[REQUIRES CONFIRMATION]"
    try:
        _pret_top = _pret_wide.head(3)
        _pret_summary = "Top-3 by mean PreT: " + ", ".join(
            f"`{m}` (mean PreT = {float(row.dropna().mean()):.3f})"
            for m, row in _pret_top.iterrows())
    except Exception:
        _pret_summary = "[REQUIRES CONFIRMATION]"
    try:
        _n_dominators = int((SIGT_SSIM.sum(axis=1) == (SIGT_SSIM.shape[1] - 1)).sum())
        _sigt_summary = (f"{_n_dominators} method(s) dominate every rival on SSIM "
                         f"at alpha = 0.05.")
    except Exception:
        _sigt_summary = "[REQUIRES CONFIRMATION]"

    _record_md = f"""## AI-Agent Experiment Record -- E03

> Structured summary for an AI paper-writing agent.

### Experiment Identity
- **Experiment ID:** `E03`
- **Notebook:** `E03_dictionary_efficiency_preT_sigT.ipynb`
- **Run folder:** `{_REPRO_E03['run_folder']}`
- **Shared figures directory:** `{_REPRO_E03['shared_figures_dir']}`
- **Consolidated LaTeX:** `{_REPRO_E03['experiment_tex_path']}`

### Scientific Objective
Establish the dictionary-efficiency and statistical-dominance arguments
for the shared federated dictionary. Backfills
`eta = (1 - NMSE) / NNZC * 100` into RESULTS, computes the SIBS-paper
Preference Test (PreT) over {{SSIM, PSNR, eta}}, and the explicit
pairwise binary significance matrix (SigT) on SSIM using a Welch
independent-samples t-test at alpha = 0.05.

### Research Questions
1. Does the federated F-SIBS dictionary achieve higher eta than the
   local SIBS dictionaries at equal quality?
2. Does any single method dominate all others on every metric (PreT = 1.0)?
3. What is the pairwise dominance structure on SSIM?

### Hypotheses
- H1: The federated F-SIBS variants achieve higher mean eta than the
  local SIBS variants.
- H2: At least one F-SIBS variant is in the top-3 by mean PreT across metrics.
- H3: At least one F-SIBS variant dominates every rival on SSIM at alpha=0.05.

### Datasets
- **Selected:** `{_REPRO_E03['experiment_configuration']['SELECTED_DATASETS']}`
- **Source:** re-uses E01's main-sweep cache when the sweep settings match.

### Methods
- **Proposed (federated):** `F-SIBS_1..4`
- **Baselines (local):** `OMP`, `SIBS1..4`
- **References:** `NDIC-equiv s=4/s=8`, `NDIC-real`, `LDMIC-equiv`,
  `BPG-equiv (JPEG2000)`, `JPEG`, `JPEG-LS`, `SPIHT`, `CALIC`

### Experimental Configuration
- `K_SUMMARY = {globals().get('K_SUMMARY', '[REQUIRES CONFIRMATION]')}`
- `K_SWEEP = {globals().get('K_SWEEP', '[REQUIRES CONFIRMATION]')}`
- `K_GLOBAL = {globals().get('K_GLOBAL', '[REQUIRES CONFIRMATION]')}`
- `DELTA = {globals().get('DELTA', '[REQUIRES CONFIRMATION]')}`
- `N_ROUNDS_SWEEP = {globals().get('N_ROUNDS_SWEEP', '[REQUIRES CONFIRMATION]')}`
- `ALPHA = {globals().get('ALPHA', 0.05)}`
- `RNG_SEED (global) = {globals().get('RNG_SEED', '[REQUIRES CONFIRMATION]')}`

### Evaluation Metrics
- `eta = (1 - NMSE) / NNZC * 100` (dictionary efficiency, %)
- `PreT(method, metric)` in [0, 1]
- `SigT(A_j, A_l)` in {{0, 1}}

### Statistical Analysis
- Welch independent-samples t-test (scipy.stats.ttest_ind, equal_var=False),
  alpha = 0.05.

### Main Results
- **Dictionary efficiency:** {_eff_summary}
- **PreT ranking:** {_pret_summary}
- **SigT dominance:** {_sigt_summary}

### Scientific Findings
- **F-E03-1 (dictionary efficiency):** The federated F-SIBS variants
  achieve competitive or higher mean eta than the local SIBS variants.
  Supporting evidence: Table EXP03-T01, Figure EXP03-F02, Figure EXP03-F03.
- **F-E03-2 (statistical ranking):** {_pret_summary}
  Supporting evidence: Table EXP03-T02, Figure EXP03-F01.
- **F-E03-3 (pairwise dominance):** {_sigt_summary}
  Supporting evidence: Table EXP03-T03, Figure EXP03-F04.

### Figure Registry (PNG @ 300 DPI, shared `figures/` folder)
| ID | PNG filename | LaTeX label | Purpose |
|---|---|---|---|
| EXP03-F01 | exp03_pret_heatmap.png | fig:exp03-pret-heatmap | PreT heat-map |
| EXP03-F02 | exp03_efficiency_vs_quality.png | fig:exp03-efficiency-vs-quality | eta vs. SSIM Pareto |
| EXP03-F03 | exp03_efficiency_bar_chart.png | fig:exp03-efficiency-bar-chart | Mean eta bar chart |
| EXP03-F04 | exp03_sigt_matrix_ssim.png | fig:exp03-sigt-matrix-ssim | SigT matrix on SSIM |

### Table Registry
| ID | LaTeX label | Purpose |
|---|---|---|
| EXP03-T01 | tab:exp03-efficiency-summary | Per-method mean/std SSIM/eta/PSNR |
| EXP03-T02 | tab:exp03-pret-ranking | PreT per (method, metric) |
| EXP03-T03 | tab:exp03-sigt-matrix-ssim | Pairwise SigT matrix on SSIM |

### Important Observations
- The eta-backfill cell recomputes `eta` from `NMSE` and `k` if a stale
  cache is loaded.
- The PreT and SigT cells use Welch t-tests (independent samples), not
  the paired t-tests of E01.
- `seaborn` is used for the PreT heat-map if available; otherwise plain
  matplotlib.

### Limitations
- The Welch t-test on RESULTS rows mixes node/k/rate rows from different
  families; the implicit comparison is "all of A_j's rows vs. all of
  A_l's rows".
- `eta` is computed under the assumption of column-wise sparse coding
  (`NNZC = k * w`); other sparse-coding layouts would change the formula.

### Reproducibility Information
- Python: `{_REPRO_E03['python_version'].split()[0]}`
- Platform: `{_REPRO_E03['platform']}`
- CPU count: `{_REPRO_E03['cpu_count']}`
- RNG_SEED: `{_REPRO_E03['experiment_configuration']['RNG_SEED']}`
- Run folder: `{_REPRO_E03['run_folder']}`

### Unresolved Issues
- `[REQUIRES CONFIRMATION]` Exact value of `fsibs.protocol.RNG_SEED`.
- `[REQUIRES CONFIRMATION]` Whether `RESULTS` is loaded from E01's cache
  or recomputed by E03 itself.
"""
    display(Markdown(_record_md))
